# Feature Engineering & Pipeline Transformation

## Business Objectives (8 Key Questions for Dashboard Visualizations):
1. **Sales & Profit Trends:** How do sales and profits perform over time (Yearly/Monthly)?
2. **Regional Profitability:** Which geographical regions drive the highest profit margins?
3. **Category Breakdown:** What are the top-performing categories and sub-categories by profit?
4. **Discount Impact:** How do varying discount levels directly affect profit margins?
5. **Logistics & SLA:** What is the average shipping duration across different ship modes, and where do delays occur?
6. **VIP Customer Identification:** Who are the Top 10 customers based on Lifetime Spend (LTV)?
7. **Order Size Distribution:** How are sales distributed across order volume tiers (Small, Medium, Bulk)?
8. **Weekly Purchasing Patterns:** Which days of the week experience the highest order volume?

In [1]:
import pandas as pd
import numpy as np
import pyarrow 

In [3]:
df = pd.read_pickle("../data/processed/superstore_cleaned.pkl")

In [4]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.959991,2,0.00,41.913601
1,2,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940002,3,0.00,219.582001
2,3,CA-2018-138688,2018-06-12,2018-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620000,2,0.00,6.871400
3,4,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.577515,5,0.45,-383.031006
4,5,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000,2,0.20,2.516400


In [30]:
def add_time_features(df):
        """
        1. Date & Logistics Features (Answers Questions 1, 5, 8)
        - Shipping Duration & Delay Status
        - Time Breakdown (Year, Month, Day, Quarter, Weekend)
        """
        
        # التواريخ متحولة ومتنظفة جاهزة من ملف Pickle!
        # مدة الشحن بالأيام مباشره
        df['Shipping_Duration'] = (df['Ship Date'] - df['Order Date']).dt.days

        # مؤشر التأخير (أكثر من 4 أيام يعتبر مخالف للـ SLA)
        df['Is_Delayed'] = (df['Shipping_Duration'] > 4).astype(int)

        # تفكيك عناصر الوقت
        df['Order_Year'] = df['Order Date'].dt.year
        df['Order_Month'] = df['Order Date'].dt.month
        df['Order_Month_Name'] = df['Order Date'].dt.month_name()
        df['Order_Day_Name'] = df['Order Date'].dt.day_name()
        df['Order_Quarter'] = df['Order Date'].dt.to_period('Q').astype(str)
        df['Is_Weekend'] = df['Order Date'].dt.dayofweek.isin([5, 6]).astype(int)

        return df


In [32]:
def add_financial_features(df):
        """
        2. Financial & Profitability Features (Answers Questions 2, 3, 4)
        - Profit Margin & Unit Price
        - Absolute Discount Amount
        """
        

        # 1️⃣ هامش الربح (Profit Margin)
        df['Profit_Margin'] = np.where(
            df['Sales'] != 0,
            df['Profit'] / df['Sales'],
            0
        )

        # 2️⃣ قيمة الخصم بالفلوس (Discount Amount)
        df['Discount_Amount'] = df['Sales'] * df['Discount']

        # 3️⃣ سعر القطعة الواحدة (Unit Price)
        df['Unit_Price'] = np.where(
            df['Quantity'] != 0,
            df['Sales'] / df['Quantity'],
            0
        )

        return df

In [33]:
def add_customer_and_order_features(df):
        """
        3. Customer & Order Segmentation Features (Answers Questions 6, 7)
        - Customer Lifetime Spend & Order Counts
        - Order Size Categorization (Small, Medium, Bulk)
        """
        print("👤 Engineering Customer Lifetime & Order Segmentation features...")

        # 1️⃣ تكرار أوردرات العميل (Customer Frequency)
        customer_counts = df.groupby('Customer ID')['Order ID'].transform('nunique')
        df['Customer_Order_Count'] = customer_counts

        # 2️⃣ إجمالي إنفاق العميل (Customer Lifetime Spend - LTV)
        customer_spend = df.groupby('Customer ID')['Sales'].transform('sum')
        df['Customer_Total_Spend'] = customer_spend

        # 3️⃣ شرائح حجم الطلبية (Order Volume Segmentation)
        df['Order_Size'] = pd.cut(
            df['Quantity'],
            bins=[0, 2, 5, np.inf],
            labels=['Small', 'Medium', 'Bulk'],
            right=True
        )

        return df

In [ ]:
def transform_all(df) -> pd.DataFrame:
         """Executes all feature engineering steps in sequence."""
                def transform_all(df) -> pd.DataFrame:
                    """Executes all feature engineering steps in sequence."""
                    df = add_time_features(df)
                    df = add_financial_features(df)
                    df = add_customer_and_order_features(df)
                    print("✨ Feature Engineering Completed Successfully!")
                    return df
                df = add_financial_features(df)
                df = add_customer_and_order_features(df)
                print("✨ Feature Engineering Completed Successfully!")
 return df

IndentationError: unindent does not match any outer indentation level (<string>, line 7)

In [18]:
# 1. تشغيل الكلاس وتطبيق كافة التحويلات
fe = FeatureEngineer(df)
df_featured = fe.transform_all()

# 2. قائمة الأعمدة الجديدة للتأكد منها
engineered_cols = [
    'Shipping_Duration', 'Is_Delayed', 'Profit_Margin',
    'Discount_Amount', 'Unit_Price', 'Customer_Total_Spend',
    'Customer_Order_Count', 'Order_Size'
]

# 3. طباعة إجمالي الأعمدة وعرض عينة من الأعمدة المصنوعة
print(f"\n📊 Total Columns now: {df_featured.shape[1]}")
df_featured[engineered_cols].head()

AttributeError: 'FeatureEngineer' object has no attribute 'transform_all'